# Importacion de librerias

In [1]:
import subprocess
import sys

def install_dependencies():
    """Instala las dependencias necesarias en el entorno de Colab."""
    packages = [
        "torch>=2.2.0",
        "transformers>=5.0.0",
        "peft>=0.18.0",
        "datasets>=3.0.0",
        "accelerate>=1.0.0",
        "bitsandbytes>=0.45.0",
        "sentencepiece>=0.2.0",
        "protobuf>=5.27.0",
        "torchao>=0.16.0",
        "unsloth[colab] @ git+https://github.com/unslothai/unsloth.git", # Added Unsloth
        "modelscope", # Added modelscope
    ]
    print("📦 Instalando dependencias...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"] + packages
    )
    print("✅ Dependencias instaladas correctamente.\n")


# Descomenta la siguiente línea si ejecutas el script completo de una vez.
# En Colab, puedes ejecutar esto en una celda aparte con:
#   !pip install -q torch transformers peft datasets accelerate bitsandbytes sentencepiece protobuf
install_dependencies()


📦 Instalando dependencias...
✅ Dependencias instaladas correctamente.



In [2]:
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ─── Verificación de GPU ───
print("=" * 60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"🟢 GPU detectada: {gpu_name} ")
else:
    print("🔴 NO se detectó GPU. Ve a Runtime > Change runtime type > T4 GPU")
    print("   El entrenamiento en CPU será extremadamente lento.")
print("=" * 60)


c:\Users\anwwe\OneDrive\Escritorio\p4-tutor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0525 14:57:40.455000 4272 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0525 14:57:40.527000 4272 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🔴 NO se detectó GPU. Ve a Runtime > Change runtime type > T4 GPU
   El entrenamiento en CPU será extremadamente lento.


# Configuracion

In [3]:
# ═══════════════════════════════════════════════════════════════
# CELDA 3: Configuración (modifica estos valores según necesites)
# ═══════════════════════════════════════════════════════════════

# ─── Modelo base ───
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# ─── Dataset ───
DATASET_PATH = "dataset.jsonl"  # Ruta al archivo JSONL

# ─── HuggingFace Token (necesario para modelos gated como Llama) ───
# Opción 1: Pegar token directamente
HF_TOKEN = ""  # Tu token de https://huggingface.co/settings/tokens
# Opción 2: Leer de archivo (si lo subiste)
TOKEN_FILE = "token_hf.txt"
if not HF_TOKEN and os.path.exists(TOKEN_FILE):
    HF_TOKEN = open(TOKEN_FILE).read().strip()

# ─── Hiperparámetros de entrenamiento ───
MAX_LENGTH = 512        # Longitud máxima de secuencia (tokens)
EPOCHS = 3              # Épocas de entrenamiento
BATCH_SIZE = 2          # Batch size por dispositivo (2 para T4, 4 para A100)
GRADIENT_ACCUMULATION = 8  # Steps de acumulación de gradientes
LEARNING_RATE = 2e-4    # Tasa de aprendizaje
WARMUP_RATIO = 0.05     # Ratio de warmup

# ─── Configuración LoRA ───
LORA_R = 16             # Rango de LoRA
LORA_ALPHA = 32         # Alpha de LoRA
LORA_DROPOUT = 0.1      # Dropout de LoRA

# ─── Cuantización ───
USE_4BIT = True         # True = QLoRA (4-bit), ideal para GPUs con poca VRAM

# ─── Directorios de salida ───
OUTPUT_DIR = "./lora-tutor"
SAVE_TO_DRIVE = False   # True para guardar también en Google Drive
DRIVE_OUTPUT = "/content/drive/MyDrive/lora-tutor"

# Login a HuggingFace


In [4]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ Autenticado en HuggingFace.\n")
else:
    print("⚠️  No se encontró token de HuggingFace.")
    print("   Si usas un modelo gated (Llama), necesitas autenticarte.")
    print("   Configura HF_TOKEN arriba o ejecuta: huggingface-cli login\n")


⚠️  No se encontró token de HuggingFace.
   Si usas un modelo gated (Llama), necesitas autenticarte.
   Configura HF_TOKEN arriba o ejecuta: huggingface-cli login



# Cargar modelo y tokenizador

In [5]:
import torch
import os
# Importar FastLanguageModel de Unsloth
from unsloth import FastLanguageModel

def load_model_and_tokenizer(model_name, use_4bit, lora_r, lora_alpha, lora_dropout):
    """Carga el modelo base con cuantización QLoRA (4-bit) o sin ella, usando Unsloth,
    y luego aplica los adaptadores LoRA."""
    print(f"📥 Cargando modelo y tokenizer con Unsloth: {model_name}")

    # Set environment variable to use Modelscope as suggested by Unsloth error message
    os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

    # Load the base model without LoRA parameters first to avoid the TypeError
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=MAX_LENGTH,
        dtype=None, # Auto-detects based on GPU availability and capability
        load_in_4bit=use_4bit,
        token=HF_TOKEN if HF_TOKEN else None,
    )

    # Now apply LoRA using get_peft_model
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Unsloth default
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none", # Unsloth default for training
        use_gradient_checkpointing=True, # Corrected to True as required by Unsloth
        random_state=3407, # Unsloth default
        max_seq_length=MAX_LENGTH, # Pass max_seq_length for Unsloth
    )

    # Configurar pad token (Unsloth might do this automatically, but good to ensure)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Unsloth ya prepara el modelo para entrenamiento en 4-bit si use_4bit=True
    model.config.use_cache = False # Deshabilitar cache para entrenamiento
    print(f"✅ Modelo cargado y adaptadores LoRA aplicados con Unsloth: {model_name}\n")
    return model, tokenizer


# Pasar los parámetros LoRA a la función de carga
model, tokenizer = load_model_and_tokenizer(
    MODEL_NAME, USE_4BIT, LORA_R, LORA_ALPHA, LORA_DROPOUT
)


c:\Users\anwwe\OneDrive\Escritorio\p4-tutor\venv\Lib\site-packages\unsloth\__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

# Aplicar adaptadores LoRA

In [ ]:
# Con Unsloth, la aplicación de LoRA se maneja directamente en FastLanguageModel.from_pretrained.
# Esta función ya no es necesaria.
# El modelo ya tiene los adaptadores LoRA aplicados al ser cargado por Unsloth.
print("🔧 Adaptadores LoRA ya aplicados durante la carga del modelo con Unsloth.")
model.print_trainable_parameters()


🔧 Adaptadores LoRA ya aplicados durante la carga del modelo con Unsloth.
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


# Preparar dataset

In [ ]:
def prepare_dataset(dataset_path, tokenizer, max_length):
    """
    Carga y tokeniza el dataset en formato chat (messages).
    Cada ejemplo tiene un campo "messages" con roles: system, user, assistant.
    Se aplica el chat template del tokenizer para formatear correctamente.
    """
    print(f"📂 Cargando dataset: {dataset_path}")
    dataset = load_dataset("json", data_files=dataset_path)

    # Verificar si el tokenizer tiene chat_template
    has_chat_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None

    def tokenize_chat(example):
        messages = example["messages"]

        if has_chat_template:
            # Usar el chat template nativo del modelo (ideal para Llama 3)
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            # Fallback: formateo manual
            parts = []
            for msg in messages:
                role = msg["role"]
                content = msg["content"]
                if role == "system":
                    parts.append(f"Sistema: {content}")
                elif role == "user":
                    parts.append(f"Usuario: {content}")
                elif role == "assistant":
                    parts.append(f"Asistente: {content}")
            text = "\n\n".join(parts) + tokenizer.eos_token

        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        tokenized["labels"] = tokenized["input_ids"].copy()
        return tokenized

    tokenized = dataset.map(
        tokenize_chat,
        remove_columns=dataset["train"].column_names,
    )

    total = len(tokenized["train"])
    print(f"✅ Dataset tokenizado: {total} ejemplos")
    print(f"   Longitud máxima: {max_length} tokens")
    print(f"   Chat template: {'Sí (nativo)' if has_chat_template else 'No (fallback manual)'}\n")
    return tokenized


tokenized = prepare_dataset(DATASET_PATH, tokenizer, MAX_LENGTH)

📂 Cargando dataset: dataset.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

✅ Dataset tokenizado: 55 ejemplos
   Longitud máxima: 512 tokens
   Chat template: Sí (nativo)



# Configurar y ejecutar entrenamiento

In [ ]:
has_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    logging_steps=5,
    save_steps=100,
    save_total_limit=2,
    fp16=has_cuda,
    bf16=False,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    optim="paged_adamw_8bit" if (USE_4BIT and has_cuda) else "adamw_torch",
    gradient_checkpointing=True,  # Ahorra VRAM a costa de velocidad
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    data_collator=data_collator,
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# Entrenar el modelo

In [ ]:
print("=" * 60)
print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 60)
print(f"   Modelo:       {MODEL_NAME}")
print(f"   Dataset:      {DATASET_PATH}")
print(f"   Épocas:       {EPOCHS}")
print(f"   Batch size:   {BATCH_SIZE} x {GRADIENT_ACCUMULATION} acum. = {BATCH_SIZE * GRADIENT_ACCUMULATION} efectivo")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   Cuantización: {'4-bit (QLoRA)' if USE_4BIT else 'Sin cuantización'}")
print(f"   Optimizador:  {'paged_adamw_8bit' if (USE_4BIT and has_cuda) else 'adamw_torch'}")
print("=" * 60 + "\n")

trainer.train()


🚀 INICIANDO ENTRENAMIENTO
   Modelo:       meta-llama/Llama-3.2-3B-Instruct
   Dataset:      dataset.jsonl
   Épocas:       3
   Batch size:   2 x 8 acum. = 16 efectivo
   Learning rate: 0.0002
   LoRA r=16, alpha=32
   Cuantización: 4-bit (QLoRA)
   Optimizador:  paged_adamw_8bit



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.902084
10,1.871186


Unsloth: Restored added_tokens_decoder metadata in ./lora-tutor/checkpoint-12/tokenizer_config.json.


TrainOutput(global_step=12, training_loss=2.243444323539734, metrics={'train_runtime': 199.7285, 'train_samples_per_second': 0.826, 'train_steps_per_second': 0.06, 'total_flos': 1441090885386240.0, 'train_loss': 2.243444323539734, 'epoch': 3.0})

# Test de inferencia

In [ ]:
def test_inference(model, tokenizer, question):
    """Ejecuta una prueba rápida de inferencia con el modelo fine-tuneado."""
    messages = [
        {
            "role": "system",
            "content": (
                "Eres un Tutor Analítico especializado en seguridad pública, "
                "violencia y análisis social en México. Tu función es asistir "
                "académicamente al usuario usando exclusivamente el contexto "
                "documental proporcionado por el sistema RAG."
            ),
        },
        {"role": "user", "content": question},
    ]

    has_chat_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    if has_chat_template:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt = (
            f"Sistema: {messages[0]['content']}\n\n"
            f"Usuario: {messages[1]['content']}\n\n"
            f"Asistente:"
        )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.2,
            top_p=0.85,
            repetition_penalty=1.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response

model.config.use_cache = True
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.eval()
print("🔄 Modelo preparado para inferencia (cache habilitado).")


# ─── Prueba con una pregunta del banco de evaluación ───
print("\n🧪 PRUEBA DE INFERENCIA:")
print("-" * 60)
test_question = "¿Cuáles son las tres entidades federativas con mayor índice de homicidios dolosos según el corpus?"
print(f"Pregunta: {test_question}\n")
answer = test_inference(model, tokenizer, test_question)
print(f"Respuesta:\n{answer}")
print("-" * 60)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔄 Modelo preparado para inferencia (cache habilitado).

🧪 PRUEBA DE INFERENCIA:
------------------------------------------------------------
Pregunta: ¿Cuáles son las tres entidades federativas con mayor índice de homicidios dolosos según el corpus?



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Respuesta:
Según los datos presentados en la sección "Índices" del documento 'Violencia Homicida', entre mayo/2018 e junio/2021, Chihuahua tiene una tasa más alta que Tamaulipas (14,9 vs 13,4), mientras que Durango supera a Sinaloa (10,7% vs 9%). En cuanto al estado mexiquense, Guadalupe está por encima de Alcoztecán (5,2% frente a 0%), pero este último sigue siendo uno de los estados menos violentos dentro del país.

Referencias:
- Documento: Violencia_Homicido.pdf 
    - Página: 15
    - Sección: Índices 

 ¿Te gustaría profundizar sobre alguna otra parte del informe o explorar otro aspecto relacionado con esta pregunta?
------------------------------------------------------------


# Eportar modelo a GGUF

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EXPORTACIÓN OPTIMIZADA GGUF PARA COLAB FREE
# Compatible con RAM limitada (~12GB)
# ═══════════════════════════════════════════════════════════════

import os
import gc
import torch

GGUF_OUTPUT = "./gguf-export"
GGUF_QUANT = "Q4_K_M"

os.makedirs(GGUF_OUTPUT, exist_ok=True)

print("=" * 60)
print("📦 EXPORTANDO GGUF (MODO COLAB FREE)")
print("=" * 60)

# ─────────────────────────────────────────────
# Limpiar memoria antes de exportar
# ─────────────────────────────────────────────
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ─────────────────────────────────────────────
# Exportar directamente con Unsloth
# ─────────────────────────────────────────────
print("\n🔧 Exportando GGUF optimizado...")

output_path = os.path.join(
    GGUF_OUTPUT,
    f"tutor-seguridad-{GGUF_QUANT.lower()}"
)

model.save_pretrained_gguf(
    output_path,
    tokenizer,
    quantization_method=GGUF_QUANT,
)

print("\n✅ Exportación completada")

# ─────────────────────────────────────────────
# Buscar archivo generado
# ─────────────────────────────────────────────
generated_files = os.listdir(GGUF_OUTPUT)

gguf_files = [f for f in generated_files if f.endswith(".gguf")]

if gguf_files:

    gguf_path = os.path.join(GGUF_OUTPUT, gguf_files[0])

    size_gb = os.path.getsize(gguf_path) / (1024**3)

    print(f"\n📦 Archivo: {gguf_files[0]}")
    print(f"📏 Tamaño: {size_gb:.2f} GB")

    try:
        from google.colab import files

        print("\n📥 Descargando GGUF...")
        files.download(gguf_path)

    except:
        print(f"\n📁 Archivo listo en: {gguf_path}")

else:
    print("❌ No se encontró el archivo GGUF")

📦 EXPORTANDO GGUF (MODO COLAB FREE)

🔧 Exportando GGUF optimizado...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in ./gguf-export/tutor-seguridad-q4_k_m/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:29<00:00, 104.66s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:01<00:00, 60.51s/it]


Unsloth: Merge process complete. Saved to `/content/gguf-export/tutor-seguridad-q4_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./gguf-export/tutor-seguridad-q4_k_m_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. Th